# Part C · Indic Token Behavior Analysis

## Pipeline Position

| Part | Notebook | GPU Required | Input | Output |
|------|----------|-------------|-------|--------|
| A | part_a_translation_evaluation | Yes (T4) | FLoRes-200 dataset | sacrebleu_results.csv |
| B | part_b_token_eda | No (CPU) | sacrebleu_results.csv | token_counts.csv, engineered_features.csv |
| **C** | **part_c_indic_token_analysis** | **No (CPU)** | **token_counts.csv** | **Vocabulary coverage charts, memory analysis** |

## Key Question

> **Does vocabulary specialisation explain the tokenisation quality gap between models?**
> 
> A model trained on 400 languages (MADLAD) has the same vocabulary size as a model trained on 200 languages (NLLB-200), yet Tamil receives a different fraction of vocabulary budget in each. Part C measures this directly by testing how each tokeniser classifies a curated set of Tamil words — from common nouns to heavily agglutinated verb phrases.

## Tamil Agglutination — Why It Challenges Tokenisers

Tamil is an agglutinative language: a single word can encode what English expresses across an entire clause:

| Tamil Word | Romanisation | English Equivalent | Morpheme Count |
|-----------|-------------|-------------------|---------------|
| மரம் | maram | tree | 1 |
| படிக்கிறாள் | padikkiraaL | she is studying | 2 |
| வந்திருக்கிறான் | vandhirukkiraaN | he has come | 4 |
| செய்துகொண்டிருக்கிறார்கள் | seydhukondirukkiraargaL | they have been doing | 6+ |

A tokeniser that has not seen enough Tamil text will split `வந்திருக்கிறான்` into 8–12 meaningless subword fragments. IndicTrans2, trained specifically on Indic languages, can often represent the same word in 2–4 tokens — each carrying real morphological meaning.

## What Part C Adds Beyond Part B

| Part B (sentence-level) | Part C (word-level) |
|------------------------|---------------------|
| Metrics aggregated over 100 full sentences | Controlled test on 20 Tamil words in 6 complexity tiers |
| expansion_ratio, avg_word_length, subword_fragmentation | Vocabulary classification: known / fragmented / unknown |
| Statistical distributions (violin plots, radar chart) | Visual token span inspection per word per model |
| Derived features (log_expansion, efficiency_score) | Memory footprint from O(n²) attention cost analysis |

## Section 1 · Environment Setup

**CPU-only notebook** — no GPU needed. Tokenisation is purely string manipulation; no tensor operations are involved.

**Why does each notebook load tokenisers independently?** Kaggle does not support kernel-to-kernel object passing — you cannot import a Python object (like a loaded tokeniser) from one notebook's runtime into another. Each notebook must call `AutoTokenizer.from_pretrained()` independently. The tokeniser weights are cached on Kaggle's filesystem after the first load, so subsequent loads in Parts B and C are fast.

**Colour palette consistency:** The `MODEL_COLORS` dictionary is duplicated across all three notebooks with identical values. This is intentional — a reviewer viewing plots from different notebooks should immediately recognise each model by colour without re-reading the legend.

In [ ]:
# ── Cell 0 · Runtime check + global visual theme ──────────────────────────────
import sys, os, warnings
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
os.makedirs("plots", exist_ok=True)

PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
MODEL_COLORS = {
    "IndicTrans2" : "#2E86AB",
    "NLLB-200"    : "#A23B72",
    "mT5"         : "#F18F01",
    "Helsinki"    : "#C73E1D",
    "MADLAD"      : "#3B1F2B",
}
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
print("\u2713 Global theme applied")

## Section 2 · Data Flow and Word-Level Test Set

**Data flow:** Part C reads `token_counts.csv` written by Part B, which itself was computed from `sacrebleu_results.csv` written by Part A. The three notebooks form a linear pipeline:

```
Part A  →  sacrebleu_results.csv  →  Part B  →  token_counts.csv  →  Part C
(GPU)          (translations)        (CPU)         (metrics)          (CPU)
```

**The sentence-level data (from Part B) is used in VIZ C2 and VIZ C3.** The word-level vocabulary coverage analysis (VIZ C1 and the token span visualiser) uses a separate curated test set of 20 Tamil words, designed to systematically stress each model across six complexity tiers:

| Tier | Description | Example |
|------|-------------|---------|
| Simple | Single-morpheme common nouns | மரம் (tree), உணவு (food) |
| Medium | 2–3 morpheme verb forms | படிக்கிறாள் (she studies) |
| High | 4+ morpheme agglutinated verbs | வந்திருக்கிறான் (he has come) |
| Domain | Government / administrative vocabulary | அரசாங்கம் (government) |
| Technical | Computing / internet terms | கணிப்பொறி (computer), இணையம் (internet) |
| Proper nouns | Tamil Nadu, Chennai, Coimbatore | தமிழ்நாடு, சென்னை |

**Why word-level analysis?** Sentence-level metrics (Part B) average over 15–30 word tokens per sentence, which hides per-word behaviour. A model may handle simple words perfectly but catastrophically fragment domain-specific or proper-noun vocabulary. The word-level test isolates these failure modes.

In [ ]:
# ── Cell 1 · Load from Part B CSV + Tokenizers ────────────────────────────────
# Part B saved token_counts.csv
# Part C loads that file and loads tokenizers only (no full model weights)
# Data flow: A → B → C via saved CSVs. No kernel sharing between notebooks.

from transformers import AutoTokenizer

token_df = pd.read_csv("../part_b_token_analysis/token_counts.csv")
print(f"✓ token_df loaded from Part B: {len(token_df)} rows")

# Load tokenizers (lightweight — no GPU needed)
TOKENIZER_IDS = {
    "IndicTrans2" : ("ai4bharat/indictrans2-en-indic-1B", {"trust_remote_code": True}),
    "NLLB-200"    : ("facebook/nllb-200-distilled-600M", {}),
    "mT5"         : ("google/mt5-base", {}),
    "Helsinki"    : ("Helsinki-NLP/opus-mt-en-dra", {}),
    "MADLAD"      : ("google/madlad400-3b-mt", {}),
}

tokenizers = {}
for name, (model_id, kwargs) in TOKENIZER_IDS.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(model_id, **kwargs)
    print(f"  ✓ {name} tokenizer loaded — vocab size: {tokenizers[name].vocab_size:,}")

## Section 3 · Vocabulary Coverage Classification

Each Tamil word is passed through each model's tokeniser and classified into one of three categories based on how many token IDs are returned:

| Class | Detection Method | Linguistic Meaning |
|-------|-----------------|-------------------|
| **Known** | `encode()` returns exactly 1 token ID (not UNK) | The word exists as a single entry in the tokeniser's vocabulary; optimal representation |
| **Fragmented** | `encode()` returns ≥ 2 token IDs, none are UNK | The word is split into subwords; meaning is distributed across pieces |
| **Unknown** | First token ID equals `tokenizer.unk_token_id` | The tokeniser has no vocabulary entry for this character sequence; falls back to a generic UNK symbol |

**Why `encode()` rather than `tokenize()`?**
`tokenize()` returns string tokens (e.g., `["வந்", "##திருக்", "கிறான்"]`). Detecting UNK from strings requires checking for the literal string `"<unk>"`, which is ambiguous — some tokenisers use different UNK spellings or have vocabulary entries that legitimately contain angle brackets. `encode()` returns integer IDs, and comparing against `tokenizer.unk_token_id` is unambiguous regardless of the UNK string representation.

**Expected findings by model:**
- **IndicTrans2**: Highest known% — 22-language Indic SentencePiece vocabulary with dedicated Tamil byte pairs
- **Helsinki**: High known% for common Tamil — EN→Dravidian multilingual vocabulary (ta/kn/ml/te) keeps all 4 scripts well represented
- **NLLB-200**: Moderate known%, higher fragmentation — 256K vocabulary spread across 200 languages
- **MADLAD-400**: Lowest known% — 256K tokens across 400 languages; Tamil receives minimal allocation
- **mT5**: Low known%, highest fragmentation — mC4 corpus is heavily English-biased; Tamil is underrepresented despite large vocabulary

In [ ]:
# ── Cell 2 · Compute vocab_stats ──────────────────────────────────────────────
# Classifies each Tamil word as:
#   known      = single token (tokenizer "knows" this word)
#   fragmented = multiple tokens (word split into subwords)
#   unknown    = maps to UNK token (tokenizer has never seen this)
#
# Tamil is agglutinative — words have many suffixes attached.
# Poor tokenizers (like mT5 trained on English-heavy data) will
# fragment even common Tamil words into many subwords.

def compute_vocab_stats(tokenizer, sample_tamil_words):
    known = fragmented = unknown = 0
    unk_id = tokenizer.unk_token_id

    for word in sample_tamil_words:
        toks = tokenizer.encode(word, add_special_tokens=False)
        if not toks:
            continue
        if unk_id and toks[0] == unk_id:
            unknown += 1
        elif len(toks) == 1:
            known += 1
        else:
            fragmented += 1

    return {"known": known, "fragmented": fragmented, "unknown": unknown}


# Tamil test words: mix of simple, agglutinated, and domain-specific
# Agglutination examples: \u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd = "he has come" (one word, many morphemes)
sample_tamil_words = [
    # Simple common words
    "\u0bae\u0bb0\u0bae\u0bcd", "\u0ba8\u0ba9\u0bcd\u0bb1\u0bbf", "\u0b89\u0ba3\u0bb5\u0bc1", "\u0ba4\u0ba3\u0bcd\u0ba3\u0bc0\u0bb0\u0bcd", "\u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bcd",
    # Medium complexity \u2014 verb forms
    "\u0baa\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb3\u0bcd", "\u0b9a\u0bc6\u0ba9\u0bcd\u0bb1\u0bbe\u0bb0\u0bcd\u0b95\u0bb3\u0bcd", "\u0baa\u0bc7\u0b9a\u0bc1\u0b95\u0bbf\u0bb1\u0bcb\u0bae\u0bcd",
    # High complexity \u2014 agglutinated
    "\u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd", "\u0b9a\u0bc6\u0baf\u0bcd\u0ba4\u0bc1\u0b95\u0bca\u0ba3\u0bcd\u0b9f\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb0\u0bcd\u0b95\u0bb3\u0bcd",
    # Domain specific
    "\u0b85\u0bb0\u0b9a\u0bbe\u0b99\u0bcd\u0b95\u0bae\u0bcd", "\u0b95\u0ba3\u0bbf\u0baa\u0bcd\u0baa\u0bca\u0bb1\u0bbf", "\u0b87\u0ba3\u0bc8\u0baf\u0bae\u0bcd", "\u0baa\u0bca\u0bb0\u0bc1\u0bb3\u0bbe\u0ba4\u0bbe\u0bb0\u0bae\u0bcd",
    # Places and proper nouns
    "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\u0ba8\u0bbe\u0b9f\u0bc1", "\u0b9a\u0bc6\u0ba9\u0bcd\u0ba9\u0bc8", "\u0b95\u0bcb\u0baf\u0bae\u0bcd\u0baa\u0bc1\u0ba4\u0bcd\u0ba4\u0bc2\u0bb0\u0bcd",
    # Rare / technical
    "\u0bae\u0b95\u0bcd\u0b95\u0bb3\u0bcd\u0ba4\u0bca\u0b95\u0bc8", "\u0bb5\u0bbf\u0bb5\u0b9a\u0bbe\u0baf\u0bbf", "\u0b9a\u0bc6\u0baf\u0bcd\u0ba4\u0bbf",
]

vocab_stats = {}
for name, tok in tokenizers.items():
    vocab_stats[name] = compute_vocab_stats(tok, sample_tamil_words)
    total = max(sum(vocab_stats[name].values()), 1)
    pct   = {k: round(v / total * 100, 1) for k, v in vocab_stats[name].items()}
    print(f"  {name:15s}: known={pct['known']}%  frag={pct['fragmented']}%  unk={pct['unknown']}%")

## Section 4 · Token Span Visualiser

**Purpose:** The visualiser renders each model's tokenisation of a single Tamil word as colour-coded HTML spans. Each distinct subword token gets a different background colour. This makes it immediately visible — without reading numbers — whether a word is represented as one token (good) or many tiny fragments (poor).

**Three complexity levels tested:**

| Level | Tamil Word | English Meaning | Expected behaviour |
|-------|-----------|----------------|-------------------|
| Simple | மரம் | tree | IndicTrans2, Helsinki likely produce 1–2 tokens; mT5/MADLAD may fragment |
| Medium | படிக்கிறாள் | she is studying | Most models produce 3–5 tokens; highly multilingual models may fragment further |
| Complex | வந்திருக்கிறான் | he has come | Reveals maximum fragmentation gaps — IndicTrans2 expected 2–4 tokens vs mT5 8–12 |

**SentencePiece ▁ stripping:** SentencePiece prepends a Unicode ▁ (U+2581, Lower One Eighth Block) to mark word boundaries (i.e., the start of a new word). The `clean_token_display()` function strips this symbol before rendering. Without stripping, spans would show `▁வந்` instead of `வந்`, making the visual cluttered and harder to read. WordPiece `##` continuation markers are also stripped for the same reason.

**How to interpret the output:**
- **One wide span** = single token; the model knows this word as a unit
- **Many small spans** = heavily fragmented; the model has no direct vocabulary entry for this Tamil morphology
- **`?` span** = UNK token; the model cannot represent this character sequence at all

In [ ]:
# ── Cell 3 · Token Span Visualizer ────────────────────────────────────────────
# Shows how each model splits a Tamil word into subword tokens.
# Strips special tokens before rendering (\u2581 SentencePiece, ## WordPiece)

def clean_token_display(tok_str):
    tok_str = tok_str.replace("\u2581", "")
    tok_str = tok_str.replace("##", "")
    tok_str = tok_str.replace("<unk>", "?")
    tok_str = tok_str.strip()
    return tok_str if tok_str else "?"


def render_token_spans(word, tokens_per_model):
    color_pool = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B",
                  "#44BBA4", "#E94F37", "#6A0572"]

    html = f"""
    <div style='font-family:monospace; margin:20px 0; padding:16px;
                border:1px solid #eee; border-radius:8px;'>
      <div style='font-size:20px; font-weight:bold; margin-bottom:14px;'>
        Word: <span style='color:#2E86AB'>{word}</span>
      </div>
    """
    for model_name, raw_tokens in tokens_per_model.items():
        tokens = [clean_token_display(t) for t in raw_tokens]
        tokens = [t for t in tokens if t]

        spans = ""
        for i, tok in enumerate(tokens):
            c = color_pool[i % len(color_pool)]
            spans += f"""
            <span style='background:{c}22; border:1.5px solid {c};
                         color:{c}; padding:3px 8px; border-radius:4px;
                         margin:2px; font-size:14px; font-weight:bold;
                         display:inline-block'>{tok}</span>"""

        count = len(tokens)
        label = f"({count} token{'s' if count != 1 else ''})"
        html += f"""
        <div style='margin:8px 0; display:flex; align-items:center; gap:12px;'>
          <span style='width:130px; font-weight:bold; font-size:13px;
                       color:#333;'>{model_name}</span>
          <span style='color:#999; font-size:12px; min-width:75px;'>{label}</span>
          <div style='flex:1;'>{spans}</div>
        </div>"""

    html += "</div>"
    return html


# Test on 3 complexity levels:
#   Simple  = one morpheme, common word
#   Medium  = 2-3 morphemes, frequent verb form
#   Complex = 4+ morphemes, heavily agglutinated verb
test_words = [
    ("Simple \u2014 \u0bae\u0bb0\u0bae\u0bcd (tree)",               "\u0bae\u0bb0\u0bae\u0bcd"),
    ("Medium \u2014 \u0baa\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb3\u0bcd (she studies)", "\u0baa\u0b9f\u0bbf\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0bb3\u0bcd"),
    ("Complex \u2014 \u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd (he has come)", "\u0bb5\u0ba8\u0bcd\u0ba4\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0bb1\u0bbe\u0ba9\u0bcd"),
]

for label, word in test_words:
    tokens_by_model = {
        name: tok.tokenize(word)
        for name, tok in tokenizers.items()
    }
    display(HTML(f"<h4>{label}</h4>"))
    display(HTML(render_token_spans(word, tokens_by_model)))

## VIZ C1 · Vocabulary Coverage Donut Charts

**What is shown:** Five donut charts — one per model — each showing the proportion of the 20-word Tamil test set that falls into **Known** (single token), **Fragmented** (multiple tokens), and **Unknown** (UNK) categories.

**Colour encoding of segments:**

| Segment | Colour | Meaning |
|---------|--------|---------|
| Known | Model's own colour (e.g., #2E86AB for IndicTrans2) | Vocabulary entry exists; word encoded as one token |
| Fragmented | #F18F01 (amber) | Word recognised but split into subwords |
| Unknown | #C73E1D (red) | Vocabulary gap; model outputs UNK |

**Why donut over a stacked bar or pie chart?**
- A **stacked bar** across 5 models is effective for comparing proportions side by side but requires the eye to track a reference line for each segment. For 3-category data per model, individual donuts make within-model proportions easier to read at a glance.
- A **pie chart** would work identically; the donut hole is purely aesthetic — it reduces ink in the centre and lets the title be placed there if needed.
- Five **separate** donuts (vs one combined chart) allow each model's 100% to stand alone, avoiding the visual ambiguity of a shared axis when comparing absolute totals (which are equal across models by design — all 20 words tested on all models).

**What to look for:** The fraction of the donut in the model's own colour (known%) is the primary quality signal. A model with 80%+ known% has strong Tamil vocabulary coverage. A model where amber (fragmented) dominates has a coverage gap but has at least seen the Tamil script. Red (unknown) segments indicate a vocabulary blind spot — the model cannot represent those Tamil character sequences at all.

In [ ]:
# ── Cell 4 · VIZ C1 · Vocabulary Coverage Donut Charts ───────────────────────
# Donut charts: what % of Tamil words each model handles well
# known = best, fragmented = acceptable, unknown = worst

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for ax, (model_name, color) in zip(axes, MODEL_COLORS.items()):
    stats = vocab_stats[model_name]
    total = max(sum(stats.values()), 1)

    sizes  = [stats["known"] / total, stats["fragmented"] / total, stats["unknown"] / total]
    colors = [color, "#F18F01", "#C73E1D"]

    wedges, _, autotexts = ax.pie(
        sizes, colors=colors, autopct="%1.0f%%",
        startangle=90, pctdistance=0.75,
        wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2),
    )
    for at in autotexts:
        at.set_fontsize(10)
        at.set_fontweight("bold")
    ax.set_title(model_name, fontsize=12, fontweight="bold", pad=10)

legend_elements = [
    mpatches.Patch(facecolor="#2E86AB", label="Known (1 token)"),
    mpatches.Patch(facecolor="#F18F01", label="Fragmented (split)"),
    mpatches.Patch(facecolor="#C73E1D", label="Unknown (UNK)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle("Tamil Vocabulary Coverage by Model", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/partc_donut_coverage.png", bbox_inches="tight", dpi=150)
plt.show()

## VIZ C2 · Transformer Memory Footprint

**The O(n²) attention problem:** Every Transformer layer computes pairwise attention between all token pairs in the sequence. For a sequence of n tokens, this requires an n × n attention matrix. Memory usage therefore scales as O(n²) — doubling the token count quadruples the memory required.

This is not a theoretical concern for Tamil: a model that fragments sentences into 2× more tokens than necessary uses **4× more attention memory** for the same input. At inference on constrained hardware (mobile devices, edge deployment), this directly limits the maximum document length the model can process.

**How the score is computed:**

| Step | Formula | Example |
|------|---------|---------|
| Per-sentence memory score | `target_token_count²` | 30 tokens → 900 units |
| Per-model score | mean over 100 sentences | avg across FLoRes-200 sample |
| Chart unit | score ÷ 1000 | expressed in k-units for readability |

**Numeric illustration (approximate):**

| Model | Avg target tokens | Memory score (token²) | Relative to IndicTrans2 |
|-------|------------------|----------------------|------------------------|
| IndicTrans2 | ~25 | ~625 | 1× (baseline) |
| Helsinki | ~28 | ~784 | 1.25× |
| NLLB-200 | ~35 | ~1,225 | 1.96× |
| mT5 | ~40 | ~1,600 | 2.56× |
| MADLAD | ~45 | ~2,025 | 3.24× |

*(Values are illustrative; actual numbers come from the 100-sentence FLoRes-200 sample.)*

**Production relevance:** For document translation or ASR post-processing (the second task in this project), the token-count difference between IndicTrans2 and MADLAD could mean the difference between processing a 2,000-word document in one batch versus requiring chunking. The memory footprint chart makes this cost concrete and model-comparable.

In [ ]:
# ── Cell 5 · VIZ C2 · Memory Footprint ───────────────────────────────────────
# Transformer attention memory scales quadratically with token count: O(n\u00b2)
# More tokens per sentence = more memory = harder to run on long texts

token_df["memory_score"] = token_df["target_token_count"] ** 2
mem_summary = token_df.groupby("model")["memory_score"].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    mem_summary.index,
    mem_summary.values / 1000,
    color=[MODEL_COLORS[m] for m in mem_summary.index],
    edgecolor="white", linewidth=1.5, height=0.55,
)
for bar, val in zip(bars, mem_summary.values / 1000):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}k", va="center", fontsize=11, fontweight="bold")

ax.set_xlabel("Relative Attention Memory Score (token\u00b2 / 1000)", fontsize=12)
ax.set_title(
    "Transformer Memory Pressure by Model\n"
    "(More tokens = O(n\u00b2) attention cost \u2014 lower is more efficient)",
    fontsize=14, fontweight="bold",
)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("plots/partc_memory_footprint.png", bbox_inches="tight", dpi=150)
plt.show()

## VIZ C3 · Characters Per Token — Tamil Subword Quality

**What this measures:** Average number of Tamil characters encoded per token, computed from `avg_word_length` in `token_counts.csv`. This is the direct inverse of subword fragmentation: a tokeniser that encodes Tamil efficiently produces **fewer, larger tokens** — each one covering more characters and more morphological content.

**Tamil script context:** Tamil is an abugida (alphasyllabary) — each base character is a consonant, optionally combined with a vowel marker diacritic. A single Tamil akshara (syllable unit) is typically 2–3 Unicode code points but reads as one phonological unit. A good tokeniser should align its subword boundaries to morpheme boundaries, not arbitrarily chop the Unicode stream.

**How to read the chart:**
- **Taller bar** = each token encodes more Tamil characters = less fragmentation = better tokeniser for Tamil
- **Error bars** (1 standard deviation over 100 sentences) show consistency — a small error bar means the tokeniser behaves predictably regardless of sentence type
- **Wide error bar** = the tokeniser's quality varies significantly across sentence types (fragmentation hot-spots for certain Tamil morphological patterns)

**The ★ Best annotation:** The model with the highest mean `avg_word_length` is highlighted with a star and blue edge. This is the tokeniser that encodes the most Tamil linguistic content per token — typically expected to be IndicTrans2 or Helsinki given their vocabulary specialisation.

**Interpreting high vs low values:**
- High chars/token (~4–6): The tokeniser has learned meaningful Tamil subword units — syllabic or morphemic chunks
- Low chars/token (~1–2): The tokeniser is splitting Tamil text almost character-by-character; this is functionally no better than byte-level tokenisation and dramatically increases sequence length

In [ ]:
# ── Cell 6 · VIZ C3 · Characters Per Token ────────────────────────────────────
# Higher avg characters per token = fewer splits = better Tamil tokenizer

chars_summary = (
    token_df.groupby("model")["avg_word_length"]
    .agg(["mean", "std"])
    .reindex(MODEL_COLORS.keys())
)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(chars_summary))
bars = ax.bar(
    x, chars_summary["mean"],
    yerr=chars_summary["std"],
    color=[MODEL_COLORS[m] for m in chars_summary.index],
    edgecolor="white", linewidth=1.5,
    capsize=5, width=0.55,
)
ax.set_xticks(x)
ax.set_xticklabels(chars_summary.index, fontsize=12)
ax.set_ylabel("Avg Characters per Token", fontsize=12)
ax.set_title(
    "Tamil Subword Quality: Characters per Token\n"
    "(Higher = less fragmentation = better Tamil tokenizer)",
    fontsize=14, fontweight="bold",
)

best_idx = chars_summary["mean"].argmax()
bars[best_idx].set_edgecolor("#2E86AB")
bars[best_idx].set_linewidth(3)
ax.text(
    best_idx,
    chars_summary["mean"].iloc[best_idx] + chars_summary["std"].iloc[best_idx] + 0.05,
    "★ Best", ha="center", color="#2E86AB", fontweight="bold",
)
plt.tight_layout()
plt.savefig("plots/partc_chars_per_token.png", bbox_inches="tight", dpi=150)
plt.show()